In [ ]:
# 1. Configure the extraction and evaluation workflow.
# Use evaluation mode for reproducible dataset runs; demo mode only enables the two inspection cells below.
MODE = "eval"  # "eval" or "demo"
DATASET = "politicause_validation"  # Frozen validation split used for prompt development.
PROMPT_NAME = "v8.7"  # Prompt matched to filtered PolitiCAUSE examples with zero or one relation.

# Model service. LM Studio is used for local checkpoints; DeepSeek uses its hosted endpoint.
LLM_PROVIDER = "lmstudio"  # Change to "deepseek" only for hosted API runs.
MODEL_NAME = "auto"  # Auto-detection is safe only when LM Studio has exactly one model loaded.
LLM_BASE_URL = "http://127.0.0.1:1234/v1"  # Use https://api.deepseek.com for DeepSeek.
LLM_API_KEY = None  # Keep credentials outside the notebook.
LLM_API_KEY_ENV = "DEEPSEEK_API_KEY"  # Preferred source for hosted runs.
LLM_API_KEY_FILE = "deepseek_api.txt"  # Local fallback; the key is read but never printed.

# Generation settings favor deterministic, complete JSON outputs.
TEMPERATURE = 0.0  # Removes sampling variance between model and RAG comparisons.
MAX_TOKENS = 8192  # Leaves enough output budget for long texts and structured triples.
CONTEXT_LENGTH = 8192  # Accommodates PolitiCAUSE text plus retrieved demonstrations.
DEEPSEEK_THINKING = "disabled"  # Enable only when evaluating a DeepSeek reasoning condition.
REASONING_EFFORT = "none"  # Explicitly disables reasoning for the local non-thinking baseline.
LLM_EXTRA_BODY = (
    {"thinking": {"type": DEEPSEEK_THINKING}}
    if LLM_PROVIDER == "deepseek" and DEEPSEEK_THINKING in {"enabled", "disabled"}
    else {}
)
LLM_TIMEOUT = 600  # Local large models can require several minutes on long inputs.
LLM_RETRY_TIMES = 3  # Retries transient server or transport failures without changing the prompt.

# Demonstration settings are ignored during evaluation mode.
DEMO_INPUT = "sample"  # Use "manual" to run DEMO_TEXT instead of dataset examples.
DEMO_TEXT = "Heavy rain caused widespread flooding in the region."
DEMO_SAMPLE_N = 3  # A small preview is sufficient for checking prompt rendering and output shape.

# Retrieval settings. Keep USE_RAG=False for the no-retrieval baseline.
USE_RAG = False
RAG_DATABASE = "politicause"  # Train-only positive support pool aligned with the target dataset.
RAG_MODE = "knn"  # Use "knn_pattern" for the semantic-plus-pattern retrieval condition.
RAG_TOP_K = 1  # Final comparisons use a single demonstration unless a k-ablation is intended.
RAG_EMBEDDING_DEVICE = "cpu"  # Avoids competing with the local language model for GPU memory.

# Evaluation settings. The subset run and full run have separate safety switches.
PRIMARY_METRIC = "anchor_window"  # Main extraction metric; strict token F1 remains auxiliary.
EVAL_SAMPLE_N = 300  # Complete frozen prompt-development subset for PolitiCAUSE.
RUN_FULL_EVAL = False  # Prevents an accidental full-dataset run after the subset finishes.
EVAL_PROGRESS_EVERY = 100  # Balances progress visibility with concise logs.
EVAL_MAX_WORKERS = 1  # Local LM Studio runs are sequential; hosted APIs may use controlled concurrency.
SAVE_EVAL_REPORT = True
REPORT_DETAIL_MODE = "errors"  # Store mistakes rather than an arbitrary prefix of correct samples.
REPORT_DETAIL_LIMIT = 200  # Caps report size while retaining all errors for typical development runs.

# Each dataset can choose the extraction metric used to classify a sample as an error.
REPORT_ERROR_METRIC_BY_DATASET = {
    "cnc": "anchor_window",
    "li": "anchor_window",
    "ade": "anchor_window",
    "causenet": "anchor_window",
    "politicause": "anchor_window",
}
REPORT_ERROR_DATASET_KEY = (
    "cnc" if DATASET.startswith("cnc")
    else "causenet" if DATASET.startswith("causenet")
    else "politicause" if DATASET.startswith("politicause")
    else DATASET
)
REPORT_ERROR_METRIC = REPORT_ERROR_METRIC_BY_DATASET.get(REPORT_ERROR_DATASET_KEY)


In [ ]:
# 2. Resolve the project environment and initialize the model client.
# The path and kernel checks prevent a similarly named installed package from masking this checkout.
# API-key resolution follows an explicit value, then an environment variable, then the local key file.
import importlib
import os
import sys
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Python executable: {sys.executable}")
if "master_thesis" not in sys.executable.lower():
    raise RuntimeError(
        "The current Notebook kernel is not Master_thesis. Change the kernel to Master_thesis, "
        "restart it, and rerun from the first cell."
    )

try:
    import openai
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "The current kernel is missing openai, so the notebook is not running in Master_thesis. "
        "Switch the kernel to Master_thesis and restart it."
    ) from exc

import json
import src.data_io as data_io

expected_data_io_path = (PROJECT_ROOT / 'src' / 'data_io.py').resolve()
actual_data_io_path = Path(data_io.__file__).resolve()
if actual_data_io_path != expected_data_io_path:
    raise RuntimeError(
        f'Imported the wrong src.data_io: {actual_data_io_path}; expected {expected_data_io_path}'
    )
data_io = importlib.reload(data_io)
if DATASET not in data_io.DATASET_FILES:
    raise RuntimeError(
        f'Dataset {DATASET!r} is not registered after reloading {actual_data_io_path}. '
        f'Available datasets: {sorted(data_io.DATASET_FILES)}'
    )
dataset_path = Path(data_io.DATA_DIR) / data_io.DATASET_FILES[DATASET]
if not dataset_path.is_file():
    raise FileNotFoundError(f'Dataset file does not exist: {dataset_path}')
load_dataset = data_io.load_dataset
print(f'Data loader: {actual_data_io_path} | dataset: {DATASET} -> {dataset_path}')
from src.eval_pipeline import EvalRunConfig, run_stream_eval
from src.evaluator import Evaluator, build_sample_judgement
from src.generator import generate, parse_output
from src.llm_client import LLMClient
from src.retriever import create_retriever
from tqdm.auto import tqdm

def read_api_key_file(path_value: str | None) -> str | None:
    if not path_value:
        return None
    key_path = Path(path_value)
    if not key_path.is_absolute():
        key_path = PROJECT_ROOT / key_path
    if not key_path.exists():
        return None
    for raw_line in key_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" in line:
            name, value = line.split("=", 1)
            if name.strip() in {"DEEPSEEK_API_KEY", "LLM_API_KEY", "API_KEY"}:
                return value.strip().strip('"').strip("'")
            continue
        return line.strip('"').strip("'")
    return None

resolved_api_key = LLM_API_KEY
api_key_source = "LLM_API_KEY"
if resolved_api_key is None and LLM_API_KEY_ENV:
    resolved_api_key = os.environ.get(LLM_API_KEY_ENV)
    api_key_source = f"env:{LLM_API_KEY_ENV}"
if resolved_api_key is None:
    resolved_api_key = read_api_key_file(LLM_API_KEY_FILE)
    api_key_source = f"file:{LLM_API_KEY_FILE}"
if not resolved_api_key:
    if LLM_PROVIDER == "deepseek":
        raise RuntimeError(
            f"Set environment variable {LLM_API_KEY_ENV} or put the key in {LLM_API_KEY_FILE} "
            "before running DeepSeek eval."
        )
    resolved_api_key = "lm-studio"
    api_key_source = "lmstudio-default"

client = LLMClient(
    provider=LLM_PROVIDER,
    base_url=LLM_BASE_URL,
    model=MODEL_NAME,
    api_key=resolved_api_key,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    context_length=CONTEXT_LENGTH,
    reasoning_effort=REASONING_EFFORT,
    extra_body=LLM_EXTRA_BODY,
    timeout=LLM_TIMEOUT,
    retry_times=LLM_RETRY_TIMES,
)

if LLM_PROVIDER == "lmstudio" and MODEL_NAME == "auto":
    available_models = client.list_loaded_models()
    if len(available_models) == 1:
        client.model = available_models[0]
        model_source = "LM Studio auto"
    elif len(available_models) == 0:
        raise RuntimeError(
            "MODEL_NAME='auto', but LM Studio /api/v0/models returned no loaded chat model. "
            "Load one model or set MODEL_NAME manually."
        )
    else:
        raise RuntimeError(
            "MODEL_NAME='auto', but LM Studio has multiple loaded models: "
            f"{available_models}. Set MODEL_NAME manually."
        )
elif MODEL_NAME == "auto":
    raise RuntimeError("MODEL_NAME='auto' is only valid for LLM_PROVIDER='lmstudio'. Set a DeepSeek model explicitly.")
else:
    model_source = f"notebook {LLM_PROVIDER}"

print(f"Provider: {client.provider}")
print(f"Base URL: {client.base_url}")
print(f"Model: {client.model}")
print(f"Model source: {model_source}")
print(f"API key source: {api_key_source}")
print(f"Max tokens: {client.max_tokens}")
print(f"Reasoning effort: {client.reasoning_effort}")
print(f"Extra body: {client.extra_body}")
print(f"Eval max workers: {EVAL_MAX_WORKERS}")
print(
    f"Report details: mode={REPORT_DETAIL_MODE}, limit={REPORT_DETAIL_LIMIT}, "
    f"error_metric={REPORT_ERROR_METRIC or 'dataset primary metric'}"
)
print(f"Timeout: {client.timeout}s | Retry times: {client.retry_times}")

def make_eval_config() -> EvalRunConfig:
    return EvalRunConfig(
        project_root=PROJECT_ROOT,
        model=client.model,
        dataset=DATASET,
        prompt_name=PROMPT_NAME,
        use_rag=USE_RAG,
        rag_mode=RAG_MODE,
        rag_top_k=RAG_TOP_K,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        primary_metric=PRIMARY_METRIC,
        progress_every=EVAL_PROGRESS_EVERY,
        max_workers=EVAL_MAX_WORKERS,
        llm_provider=client.provider,
        llm_base_url=client.base_url,
        context_length=client.context_length,
        reasoning_effort=client.reasoning_effort,
        llm_extra_body=client.extra_body,
        api_key_source=api_key_source,
        save_report=SAVE_EVAL_REPORT,
        report_detail_mode=REPORT_DETAIL_MODE,
        report_detail_limit=REPORT_DETAIL_LIMIT,
        report_error_metric=REPORT_ERROR_METRIC,
        metadata_path=globals().get("bge_metadata_path"),
        embeddings_path=globals().get("bge_embeddings_path"),
    )


In [ ]:
# 3. Test the configured model service before loading retrieval resources or starting an evaluation.
# The hosted-service probe also confirms that strict JSON can be parsed by the production parser.
if LLM_PROVIDER == "lmstudio":
    assert client.ping(), "LM Studio server is not running or is unreachable."
    print("LM Studio connection OK")
elif LLM_PROVIDER == "deepseek":
    smoke_raw = client.chat([
        {"role": "system", "content": "Output strict JSON only."},
        {"role": "user", "content": 'Return exactly {"has_causal": false, "triples": []}.'},
    ])
    smoke_parsed = parse_output(smoke_raw)
    assert smoke_parsed == {"has_causal": False, "triples": []}
    print(f"DeepSeek connection OK: {client.model}")
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")


In [ ]:
# 4. Resolve and verify the retrieval cache selected by RAG_DATABASE.
# This check is kept separate so cache-path problems are visible before prompt construction.
from pathlib import Path
from src.retriever import resolve_rag_cache_paths

bge_metadata_path, bge_embeddings_path = resolve_rag_cache_paths(RAG_DATABASE)
assert bge_metadata_path.exists(), f"BGE metadata cache does not exist: {bge_metadata_path}"
assert bge_embeddings_path.exists(), f"BGE embeddings cache does not exist: {bge_embeddings_path}"
print(f"BGE metadata cache is ready: {bge_metadata_path}")
print(f"BGE embeddings cache is ready: {bge_embeddings_path}")
print(f"RAG database: {RAG_DATABASE} | mode: {RAG_MODE} | USE_RAG={USE_RAG} | top_k={RAG_TOP_K}")


In [ ]:
# 5. Select a representative input and render the zero-shot prompt without calling the model.
# Printing several candidates first makes it explicit which sample supplies the preview text.
import importlib

import src.prompt_builder as prompt_builder
from src.data_io import load_dataset

prompt_builder = importlib.reload(prompt_builder)
build_messages = prompt_builder.build_messages

if DEMO_INPUT == "sample":
    rag_samples = load_dataset(DATASET, n=DEMO_SAMPLE_N)
    print(f"Previewing {len(rag_samples)} samples; the first sample is used to render the prompt")
    for sample in rag_samples:
        print(f"- id={sample['id']}: {sample['text']}")
    RAG_INPUT_TEXT = rag_samples[0]["text"]
    RAG_INPUT_ID = rag_samples[0]["id"]
else:
    RAG_INPUT_TEXT = DEMO_TEXT
    RAG_INPUT_ID = None

print(f"\nPrompt input id={RAG_INPUT_ID}: {RAG_INPUT_TEXT}")
messages = build_messages(RAG_INPUT_TEXT, use_rag=False, retriever=None, top_k=0, prompt_name=PROMPT_NAME)
for m in messages:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


In [ ]:
# 6. Initialize the selected retriever and render the corresponding retrieval-augmented prompt.
# When USE_RAG is false, the same code path still exposes the baseline prompt for comparison.
import importlib

import src.prompt_builder as prompt_builder
import src.retriever as retriever_module

prompt_builder = importlib.reload(prompt_builder)
retriever_module = importlib.reload(retriever_module)
build_messages = prompt_builder.build_messages
create_retriever = retriever_module.create_retriever

retriever = None
if USE_RAG:
    retriever = create_retriever(
        RAG_MODE,
        metadata_path=bge_metadata_path,
        embeddings_path=bge_embeddings_path,
        embedding_device=RAG_EMBEDDING_DEVICE,
    )

messages_rag = build_messages(
    RAG_INPUT_TEXT,
    use_rag=USE_RAG,
    retriever=retriever,
    top_k=RAG_TOP_K,
    rag_mode=RAG_MODE,
    prompt_name=PROMPT_NAME,
)
for m in messages_rag:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


In [ ]:
# 7. Inspect the demonstrations returned for the preview input.
# Review relevance and label structure here before paying the cost of a model evaluation.
if retriever is None:
    print("Retrieval is disabled. Set USE_RAG=True and select RAG_MODE to test it.")
else:
    examples = retriever.retrieve(RAG_INPUT_TEXT, top_k=RAG_TOP_K)
    print(f"Retrieved {len(examples)} examples:")
    for i, ex in enumerate(examples, 1):
        print(f"\n--- Example {i} ---")
        print(ex)


In [ ]:
# 8. Exercise structured-output parsing against common response wrappers.
# These fixed cases cover plain JSON, prose prefixes, Markdown fences, and reasoning tags without model calls.
import json
from src.generator import generate, parse_output
from src.data_io import load_dataset

test_cases = [
    '{"has_causal": true, "triples": []}',
    'Here is the result:\n{"has_causal": false, "triples": []}',
    '```json\n{"has_causal": true, "triples": []}\n```',
    'Sure!\n```\n{"has_causal": false, "triples": []}\n```\nDone.',
    '<think>reasoning</think>\n{"has_causal": true, "triples": []}',
]
for i, tc in enumerate(test_cases, 1):
    try:
        result = parse_output(tc)
        print(f"Case {i} passed: {result}")
    except Exception as e:
        print(f"Case {i} failed to parse: {e}")


In [ ]:
# 9. Run one manually supplied example when MODE="demo" and DEMO_INPUT="manual".
if MODE == "demo" and DEMO_INPUT == "manual":
    result = generate(
        text=DEMO_TEXT,
        sample_id=None,
        client=client,
        retriever=retriever if USE_RAG else None,
        use_rag=USE_RAG,
        top_k=RAG_TOP_K,
        rag_mode=RAG_MODE,
        prompt_name=PROMPT_NAME,
    )
    print("=== Single-example output ===")
    print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# 10. Run a small dataset preview when MODE="demo" and DEMO_INPUT="sample".
# Gold and predicted structures are printed together to make schema or span errors easy to spot.
if MODE == "demo" and DEMO_INPUT == "sample":
    samples = load_dataset(DATASET, n=DEMO_SAMPLE_N)
    for s in samples:
        result = generate(
            text=s["text"],
            sample_id=s["id"],
            client=client,
            retriever=retriever if USE_RAG else None,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
            prompt_name=PROMPT_NAME,
        )
        print(f"\n--- id={s['id']} ---")
        print(f"Input: {s['text']}")
        print("Gold:")
        print(json.dumps({"has_causal": s["has_causal"], "triples": s["relations"]}, indent=2, ensure_ascii=False))
        print("Pred:")
        print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# 11. Validate span normalization and token-level F1 on transparent edge cases.
# This is a local metric check and does not use the configured language model.
from src.evaluator import Evaluator, build_sample_judgement, match_triples, preprocess_span, token_f1

test_pairs = [
    ("A super heavy rain", "super heavy rain"),
    ("The strike resumed on Tuesday.", "strike resumed on tuesday"),
    ("  Multiple   spaces  ", "multiple spaces"),
]
for raw, expected in test_pairs:
    got = preprocess_span(raw)
    mark = "OK" if got == expected else "FAIL"
    print(f"{mark} {raw!r} -> {got!r} (expected {expected!r})")

print(f"exact match: {token_f1('heavy rain', 'heavy rain'):.3f}")
print(f"no overlap: {token_f1('apple pie', 'rocket science'):.3f}")
print(f"article difference: {token_f1('a super heavy rain', 'the super heavy rain'):.3f}")
print(f"partial overlap: {token_f1('heavy rain caused flooding', 'heavy rain'):.3f}")


In [ ]:
# 12. Validate aggregation and error accounting with fixed predictions and labels.
# The examples deliberately include a correct relation, a correct negative, and a false positive.
mock_preds = [
    {"id": 1, "has_causal": True, "triples": [
        {"cause": {"span": "heavy rain"}, "relation": "caused", "effect": {"span": "flooding"}}
    ]},
    {"id": 2, "has_causal": False, "triples": []},
    {"id": 3, "has_causal": True, "triples": [
        {"cause": {"span": "the strike"}, "relation": "caused", "effect": {"span": "delays"}}
    ]},
]
mock_golds = [
    {"id": 1, "has_causal": True, "relations": [{"cause": "heavy rain", "effect": "the flooding"}]},
    {"id": 2, "has_causal": False, "relations": []},
    {"id": 3, "has_causal": False, "relations": []},
]

mock_evaluator = Evaluator()
for pred, gold in zip(mock_preds, mock_golds):
    mock_evaluator.update(prediction=pred, gold=gold)
print(mock_evaluator.format_report(title="SPEC_05 mock evaluator report"))


In [ ]:
# 13. Review the metadata that will identify the saved evaluation report.
# Confirming this label before execution avoids mixing models, prompts, or retrieval conditions.
eval_config = make_eval_config()
print(
    "Eval report metadata: "
    f"model={eval_config.model} | dataset={eval_config.dataset} | "
    f"prompt={eval_config.prompt_name} | "
    f"rag={'off' if not eval_config.use_rag else eval_config.rag_mode + '-k' + str(eval_config.rag_top_k)}"
)


In [ ]:
# 14. Run the configured evaluation subset.
# EVAL_SAMPLE_N controls this development run; progress and checkpoint handling live in run_stream_eval.
if MODE == "eval":
    import importlib
    import src.data_io as current_data_io
    if DATASET not in current_data_io.DATASET_FILES:
        current_data_io = importlib.reload(current_data_io)
    if DATASET not in current_data_io.DATASET_FILES:
        raise RuntimeError(
            f'Dataset {DATASET!r} is still unregistered after reloading '
            f'{Path(current_data_io.__file__).resolve()}.'
        )
    eval_config = make_eval_config()
    eval_samples = current_data_io.load_dataset(DATASET, n=EVAL_SAMPLE_N)
    small_eval_report = run_stream_eval(
        eval_samples,
        label=f"{DATASET} first {len(eval_samples)}",
        client=client,
        config=eval_config,
        existing_retriever=globals().get("retriever"),
        progress_factory=tqdm,
        emit=print,
    )
else:
    print("MODE is not eval. Set MODE = 'eval' at the top and rerun the notebook to evaluate real data.")


In [ ]:
# 15. Optionally run the full dataset after the subset has been inspected.
# RUN_FULL_EVAL is intentionally independent so rerunning the notebook cannot start it by accident.
if MODE == "eval" and RUN_FULL_EVAL:
    eval_config = make_eval_config()
    full_samples = load_dataset(DATASET)
    full_eval_report = run_stream_eval(
        full_samples,
        label=f"{DATASET} full",
        client=client,
        config=eval_config,
        existing_retriever=globals().get("retriever"),
        progress_factory=tqdm,
        emit=print,
    )
else:
    print("Full eval did not run. Set MODE = 'eval' and RUN_FULL_EVAL = True to enable it.")


In [ ]:
# 16. Compare one extraction-error group from a saved report.
# Select the report, metric layer, and bucket explicitly so the printed examples answer one question at a time.
import importlib
from pathlib import Path

import src.eval_pipeline as eval_pipeline

eval_pipeline = importlib.reload(eval_pipeline)
format_eval_diff_examples = eval_pipeline.format_eval_diff_examples
load_sample_judgements_from_report = eval_pipeline.load_sample_judgements_from_report

DIFF_REPORT_NAME = "qwen-qwen3.6-35b-a3b_cnc_positive-n200_prompt-v11_rag-off_20260718-024634"  # Saved run to inspect.
DIFF_N = 40  # Large enough to reveal recurring errors without flooding the notebook.
DIFF_REPORT_DIR = Path(PROJECT_ROOT) / "results" / "eval_report"

DIFF_GROUPS = [

     ("detected_only", "strict_token_f1", "fp"),

]

try:
    diff_sample_judgements = load_sample_judgements_from_report(
        DIFF_REPORT_NAME,
        report_dir=DIFF_REPORT_DIR,
    )
except FileNotFoundError as exc:
    print(exc)
else:
    print(f"Loaded {len(diff_sample_judgements)} sample details from {DIFF_REPORT_NAME}")
    print("Note: if the report was capped, this only analyzes the sample details saved in the report.")
    print()
    for layer, metric, bucket in DIFF_GROUPS:
        print(
            format_eval_diff_examples(
                diff_sample_judgements,
                layer=layer,
                metric=metric,
                bucket=bucket,
                limit=DIFF_N,
                title=f"{DIFF_REPORT_NAME} {layer}/{metric}/{bucket.upper()}",
            )
        )
        print()


In [ ]:
# 17. Inspect detection errors independently of relation-span matching.
# This view isolates false causal decisions before considering extraction quality.
import importlib
from pathlib import Path

import src.eval_pipeline as eval_pipeline

eval_pipeline = importlib.reload(eval_pipeline)
format_detection_diff_examples = eval_pipeline.format_detection_diff_examples
load_sample_judgements_from_report = eval_pipeline.load_sample_judgements_from_report

DETECTION_REPORT_NAME = globals().get(
    "DIFF_REPORT_NAME",
    "qwen-qwen3.5-35b-a3b_cnc-n300_prompt-v9.6_rag-off_20260709-002600",
)
DETECTION_N = 40  # Keep the inspection depth consistent with the extraction-error view.
DETECTION_REPORT_DIR = Path(PROJECT_ROOT) / "results" / "eval_report"

DETECTION_BUCKETS = ["fp"]

try:
    detection_sample_judgements = load_sample_judgements_from_report(
        DETECTION_REPORT_NAME,
        report_dir=DETECTION_REPORT_DIR,
    )
except FileNotFoundError as exc:
    print(exc)
else:
    print(f"Loaded {len(detection_sample_judgements)} sample details from {DETECTION_REPORT_NAME}")
    print("Note: if the report was capped, this only analyzes the sample details saved in the report.")
    print()
    for bucket in DETECTION_BUCKETS:
        print(
            format_detection_diff_examples(
                detection_sample_judgements,
                bucket=bucket,
                limit=DETECTION_N,
                title=f"{DETECTION_REPORT_NAME} detection/{bucket.upper()}",
            )
        )
        print()
